# MAP MOD-00 — Homologacion de Municipios Colombia (DIVIPOLA)

**Proyecto:** Medicion del Impacto de la Variabilidad Climatica en Produccion Arrocera
**Maestria en Ciencia de Datos — Pontificia Universidad Javeriana Cali**
**Directora:** Yady Tatiana Solano Correa
**Equipo:** Rafael Del Castillo Pavajeau | Wilson Brinez Bermudez | Martin Naranjo Escobar

---

## Descripcion del modulo

Este modulo es el **cimiento de la arquitectura modular v2.0**. Construye la **Tabla Maestra de Homologacion de Municipios**: mapea los nombres de municipio de cada fuente al codigo DIVIPOLA oficial del DANE.

### Por que es necesario?
Cada fuente usa convenciones distintas para el mismo municipio. Sin homologacion, el JOIN en MOD-06 pierde municipios silenciosamente → dataset de solo ~170 observaciones. Con `cod_divipola` como llave universal, el dataset puede crecer a 600-3000+ observaciones.

---

## Estructura de archivos

```
Proyecto_Aplicado_II/
├── notebooks/
│   └── MOD00_homologacion_municipios.ipynb
├── data/
│   ├── raw/
│   │   └── 00_raw_DIVIPOLA.xlsx
│   └── processed/
│       ├── 00_DF_Homologacion_Municipios.xlsx    <- 3 hojas
│       ├── 00_DF_Homologacion_Municipios.parquet <- para joins
│       └── 00_Reporte_Cobertura.xlsx
└── backup/
```

---

## Entradas / Salidas

| Tipo | Descripcion |
|------|-------------|
| **Entrada 1** | API DANE DIVIPOLA — 1.122 municipios oficiales |
| **Entrada 2** | API datos.gov.co `hp9r-jxuu` — catalogo estaciones IDEAM |
| **Entrada 3** | URL UPRA — nombres de municipios agricolas (arroz riego) |
| **Entrada 4** | APIs IDEAM (`sbwg-7ju4`, `s54a-sgyg`, `uext-mhny`) — muestras |
| **Salida 1** | `data/raw/00_raw_DIVIPOLA.xlsx` |
| **Salida 2** | `data/processed/00_DF_Homologacion_Municipios.xlsx` (3 hojas) |
| **Salida 3** | `data/processed/00_DF_Homologacion_Municipios.parquet` |
| **Salida 4** | `data/processed/00_Reporte_Cobertura.xlsx` |

---

## Posicion en la cadena modular

```
[MOD-00] -> cod_divipola -> MOD-01, MOD-02, MOD-03, MOD-04, MOD-05, MOD-06
```

**EJECUTAR ESTE MODULO ANTES QUE CUALQUIER OTRO**

---
## CELDA 1 — Instalacion de dependencias y configuracion de entorno

In [ ]:
# ============================================================
# MOD-00 | Homologacion de Municipios Colombia (DIVIPOLA)
# Version : 1.0  —  Arquitectura modular v2.0
# ============================================================
!pip install openpyxl rapidfuzz pyarrow requests -q

import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import os, re, time, warnings
from io import BytesIO
from rapidfuzz import fuzz, process as fuzz_process

warnings.filterwarnings('ignore')

from google.colab import drive
drive.mount('/content/drive')

print('Entorno configurado correctamente')
print('   rapidfuzz : OK')


---
## CELDA 2 — Definicion de rutas y parametros

> Verifica que `RUTA_BASE` coincida con la carpeta en tu Google Drive.

In [ ]:
# ============================================================
# CONFIGURACION DE RUTAS Y PARAMETROS — MOD-00
# ============================================================

# Ajusta esta ruta si tu carpeta en Drive tiene otro nombre
RUTA_BASE      = '/content/drive/MyDrive/Colab Notebooks/Proyecto_Aplicado_II'

RUTA_RAW       = os.path.join(RUTA_BASE, 'data', 'raw')
RUTA_PROCESSED = os.path.join(RUTA_BASE, 'data', 'processed')
RUTA_BACKUP    = os.path.join(RUTA_BASE, 'backup')

for ruta in [RUTA_RAW, RUTA_PROCESSED, RUTA_BACKUP]:
    os.makedirs(ruta, exist_ok=True)

ARCHIVO_DIVIPOLA_RAW  = os.path.join(RUTA_RAW,       '00_raw_DIVIPOLA.xlsx')
ARCHIVO_HOMOLOG_XLSX  = os.path.join(RUTA_PROCESSED,  '00_DF_Homologacion_Municipios.xlsx')
ARCHIVO_HOMOLOG_PARQ  = os.path.join(RUTA_PROCESSED,  '00_DF_Homologacion_Municipios.parquet')
ARCHIVO_REPORTE       = os.path.join(RUTA_PROCESSED,  '00_Reporte_Cobertura.xlsx')

URL_DIVIPOLA    = 'https://www.dane.gov.co/files/censo2018/nomenclatura/CNT_DIVIPOLA_Municipios.xlsx'
URL_ESTACIONES  = 'https://www.datos.gov.co/resource/hp9r-jxuu.json'
URL_EVA         = 'https://upra.gov.co/sites/default/files/2025-07/20250617_BaseAgricola20192024.xlsx'
INI_FILA_EVA    = 8

APIS_IDEAM = {
    'temperatura'  : 'https://www.datos.gov.co/resource/sbwg-7ju4.json',
    'precipitacion': 'https://www.datos.gov.co/resource/s54a-sgyg.json',
    'humedad'      : 'https://www.datos.gov.co/resource/uext-mhny.json',
}

LIMIT_MUESTRA     = 100
LIMIT_ESTACIONES  = 20000
UMBRAL_FUZZY_AUTO = 85
UMBRAL_FUZZY_MAN  = 70

CULTIVO_UPRA  = 'ARROZ'
DESAGR_UPRA   = 'RIEGO'
ANIO_INI_UPRA = 2020
ANIO_FIN_UPRA = 2024

print('Rutas configuradas')
print(f'   Raw       -> {RUTA_RAW}')
print(f'   Processed -> {RUTA_PROCESSED}')


---
## CELDA 3 — Funciones utilitarias

In [ ]:
# ============================================================
# CELDA 3 — Funciones utilitarias
# Correccion v2: homologar_nombres garantiza strings puros
# antes de pasarlos a rapidfuzz (evita TypeError numpy.int64)
# ============================================================

def normalizar_texto(serie):
    return (serie.astype(str).str.strip().str.upper()
                 .str.replace(r'\s+', ' ', regex=True))


def fix_encoding(x):
    if isinstance(x, str):
        try:
            return x.encode('latin1').decode('utf8')
        except (UnicodeDecodeError, UnicodeEncodeError):
            return x
    return x


def reporte_df(df, nombre='DataFrame'):
    print(f'\n{"="*58}\n  {nombre}\n{"="*58}')
    print(f'  Filas    : {df.shape[0]:,}')
    print(f'  Columnas : {df.shape[1]}')
    nulos = df.isna().sum()
    nulos_c = nulos[nulos > 0]
    if nulos_c.empty:
        print('  Nulos    : OK - Sin valores nulos')
    else:
        print(f'  Nulos    : {len(nulos_c)} cols con nulos')
        for c, n in nulos_c.items():
            print(f'             - {c}: {n:,} ({n/len(df)*100:.1f}%)')
    print(f'{"="*58}\n')


def guardar_excel(df_o_dict, ruta, nombre_log=''):
    etiqueta = nombre_log or os.path.basename(ruta)
    if isinstance(df_o_dict, dict):
        with pd.ExcelWriter(ruta, engine='openpyxl') as w:
            for hoja, df in df_o_dict.items():
                df.to_excel(w, sheet_name=hoja, index=False)
        total = sum(d.shape[0] for d in df_o_dict.values())
        print(f'OK {etiqueta}  ->  {ruta}')
        print(f'   Hojas: {list(df_o_dict.keys())} | Total: {total:,} filas\n')
    else:
        df_o_dict.to_excel(ruta, index=False, engine='openpyxl')
        print(f'OK {etiqueta}  ->  {ruta}')
        print(f'   Shape: {df_o_dict.shape[0]:,} x {df_o_dict.shape[1]}\n')


def homologar_nombres(nombres_fuente, df_div, umbral_auto=85, umbral_manual=70):
    '''
    Fuzzy matching entre nombres de una fuente y el catalogo DIVIPOLA.

    Usa token_sort_ratio de rapidfuzz contra la clave compuesta
    NOMBRE_DEPTO|NOMBRE_OFICIAL para evitar homonimos entre departamentos.

    IMPORTANTE: Todos los candidatos se convierten explicitamente a str
    antes de pasar a rapidfuzz para evitar TypeError con numpy.int64.

    Parametros
    ----------
    nombres_fuente : list  — nombres unicos de la fuente a homologar
    df_div         : DataFrame con cod_divipola, nombre_oficial, nombre_depto
    umbral_auto    : int   — umbral minimo para homologacion automatica (85)
    umbral_manual  : int   — umbral minimo para revision manual (70)

    Retorna
    -------
    DataFrame con:
      nombre_fuente | nombre_divipola | cod_divipola | nombre_depto | score | metodo
    '''
    # ── Construir lista de candidatos como strings puros ────────
    # Convertir EXPLICITAMENTE a str para evitar TypeError de numpy tipos
    candidatos_comp = [
        str(d) + '|' + str(m)
        for d, m in zip(
            df_div['nombre_depto'].fillna('').astype(str),
            df_div['nombre_oficial'].fillna('').astype(str)
        )
    ]

    # Indice inverso: candidato_comp -> fila de df_div
    idx_map = {c: i for i, c in enumerate(candidatos_comp)}

    resultados = []

    for nombre in nombres_fuente:
        # Garantizar que el query tambien es str puro
        nombre_str = str(nombre).strip().upper()
        if not nombre_str or nombre_str in ('', 'NAN', 'NONE', 'NAT'):
            continue

        try:
            match_comp, score, _ = fuzz_process.extractOne(
                nombre_str,
                candidatos_comp,
                scorer=fuzz.token_sort_ratio
            )
        except Exception as e:
            print(f'  AVISO: error en matching para {nombre_str!r}: {e}')
            resultados.append({
                'nombre_fuente': nombre_str, 'nombre_divipola': None,
                'cod_divipola': None, 'nombre_depto': None,
                'score': 0, 'metodo': 'Error'
            })
            continue

        # Recuperar fila de df_div usando el indice
        fila_idx = idx_map.get(match_comp)
        if fila_idx is None:
            resultados.append({
                'nombre_fuente': nombre_str, 'nombre_divipola': None,
                'cod_divipola': None, 'nombre_depto': None,
                'score': score, 'metodo': 'Sin_match'
            })
            continue

        fila = df_div.iloc[fila_idx]

        # Determinar metodo segun score
        mun_m = str(fila['nombre_oficial'])
        if nombre_str == mun_m or score == 100:
            metodo = 'Exacto'
        elif score >= umbral_auto:
            metodo = f'Fuzzy_{umbral_auto}'
        elif score >= umbral_manual:
            metodo = 'Revision_manual'
        else:
            metodo = 'Sin_match'

        resultados.append({
            'nombre_fuente'  : nombre_str,
            'nombre_divipola': str(fila['nombre_oficial']),
            'cod_divipola'   : str(fila['cod_divipola']),
            'nombre_depto'   : str(fila['nombre_depto']),
            'score'          : round(float(score), 1),
            'metodo'         : metodo
        })

    return pd.DataFrame(resultados)


print('OK Funciones utilitarias cargadas')
print('   - normalizar_texto()')
print('   - fix_encoding()')
print('   - reporte_df()')
print('   - guardar_excel()')
print('   - homologar_nombres()  [v2 - robusto a tipos numpy]')


---
## CELDA 4 — Descarga del listado DIVIPOLA oficial (DANE)

In [ ]:
# ============================================================
# CELDA 4 — Descarga del listado DIVIPOLA (multiples fuentes)
# Estrategia de fallback en 3 intentos:
#   1. API Socrata datos.gov.co  (recurso gdxc-w37w) — SIN descarga de archivo
#   2. Geoportal DANE            (Excel directo)
#   3. URL DANE alternativa      (censo nomenclatura)
# ============================================================

import requests, pandas as pd, os, time
from io import BytesIO

df_divipola_raw = None
FUENTE_USADA    = None

# ── Estrategia 1: API Socrata datos.gov.co (gdxc-w37w) ──────
# Esta es la fuente MAS CONFIABLE: no depende de un archivo Excel descargable.
# La API devuelve JSON con campos cod_depto, nombre_depto, cod_municipio, nombre_municipio.
# El recurso gdxc-w37w es el dataset oficial DIVIPOLA publicado por el DANE en datos.gov.co.
print('Intento 1: API Socrata datos.gov.co (recurso gdxc-w37w)...')
URL_SOCRATA = 'https://www.datos.gov.co/resource/gdxc-w37w.json'
try:
    registros = []
    offset = 0
    limit  = 1000
    while True:
        resp = requests.get(
            URL_SOCRATA,
            params={'$limit': limit, '$offset': offset},
            timeout=60
        )
        resp.raise_for_status()
        chunk = resp.json()
        if not chunk:
            break
        registros.extend(chunk)
        offset += limit
        if len(chunk) < limit:
            break
        time.sleep(0.3)

    df_divipola_raw = pd.DataFrame(registros)
    if len(df_divipola_raw) > 100:
        FUENTE_USADA = 'API Socrata datos.gov.co (gdxc-w37w)'
        print(f'   OK Descargados {len(df_divipola_raw):,} registros')
        print(f'   Columnas: {df_divipola_raw.columns.tolist()}')
        display(df_divipola_raw.head(5))
    else:
        print(f'   AVISO: Solo {len(df_divipola_raw)} registros — intentando fuente 2')
        df_divipola_raw = None
except Exception as e:
    print(f'   AVISO Error fuente 1: {e}')
    df_divipola_raw = None

# ── Estrategia 2: Geoportal DANE (Excel) ────────────────────
if df_divipola_raw is None:
    print('\nIntento 2: Geoportal DANE (Excel)...')
    URL_GEOPORTAL = 'https://geoportal.dane.gov.co/descargas/divipola/DIVIPOLA_Municipios.xlsx'
    try:
        resp2 = requests.get(URL_GEOPORTAL, timeout=120)
        resp2.raise_for_status()
        df_divipola_raw = pd.read_excel(BytesIO(resp2.content), engine='openpyxl', header=None)
        if len(df_divipola_raw) > 100:
            FUENTE_USADA = 'Geoportal DANE Excel'
            print(f'   OK Descargados {len(df_divipola_raw):,} filas')
            print(f'   Columnas: {df_divipola_raw.columns.tolist()}')
            display(df_divipola_raw.head(8))
        else:
            df_divipola_raw = None
    except Exception as e:
        print(f'   AVISO Error fuente 2: {e}')
        df_divipola_raw = None

# ── Estrategia 3: URL DANE censo nomenclatura ────────────────
if df_divipola_raw is None:
    print('\nIntento 3: URL DANE nomenclatura...')
    URL_DANE_ALT = 'https://www.dane.gov.co/files/censo2018/nomenclatura/CNT_DIVIPOLA_Municipios.xlsx'
    try:
        resp3 = requests.get(URL_DANE_ALT, timeout=120)
        resp3.raise_for_status()
        df_divipola_raw = pd.read_excel(BytesIO(resp3.content), engine='openpyxl', header=None)
        if len(df_divipola_raw) > 100:
            FUENTE_USADA = 'URL DANE nomenclatura censo 2018'
            print(f'   OK Descargados {len(df_divipola_raw):,} filas')
            display(df_divipola_raw.head(8))
        else:
            df_divipola_raw = None
    except Exception as e:
        print(f'   AVISO Error fuente 3: {e}')
        df_divipola_raw = None

# ── Resultado ────────────────────────────────────────────────
if df_divipola_raw is not None:
    print(f'\nFUENTE EXITOSA: {FUENTE_USADA}')
    print(f'Registros totales: {len(df_divipola_raw):,}')
    # Guardar raw en Drive si fue Excel (la API Socrata no genera archivo Excel)
    if 'Excel' in str(FUENTE_USADA) or 'URL DANE' in str(FUENTE_USADA):
        try:
            if 'resp2' in dir() and FUENTE_USADA == 'Geoportal DANE Excel':
                with open(ARCHIVO_DIVIPOLA_RAW, 'wb') as f:
                    f.write(resp2.content)
            elif 'resp3' in dir():
                with open(ARCHIVO_DIVIPOLA_RAW, 'wb') as f:
                    f.write(resp3.content)
            print(f'   Raw guardado: {ARCHIVO_DIVIPOLA_RAW}')
        except:
            pass
else:
    print()
    print('ERROR: Todas las fuentes automaticas fallaron.')
    print('Por favor ejecuta la CELDA 4-MANUAL debajo de esta.')


---
## CELDA 5 — Construccion del DataFrame DIVIPOLA canonico

> Ajusta `FILA_HEADER` segun el output de la celda anterior.

In [ ]:
# ============================================================
# CELDA 5 — Construccion del DataFrame DIVIPOLA canonico
# Detecta automaticamente la estructura segun la fuente usada.
# Correccion v3: maneja correctamente el caso Excel del Geoportal DANE
# (columnas: cod_dpto, dpto, cod_mpio, nom_mpio)
# ============================================================

if df_divipola_raw is None:
    raise RuntimeError(
        'df_divipola_raw no disponible.\n'
        'Ejecuta la CELDA 4 y verifica la conectividad de red.'
    )

# ── Normalizar nombres de columna a minusculas para comparacion ──
col_map = {str(c).lower().strip(): c for c in df_divipola_raw.columns}
cols_lower = list(col_map.keys())
print(f'Columnas disponibles: {df_divipola_raw.columns.tolist()}')
print(f'Total filas raw: {len(df_divipola_raw):,}')
print()

# ── Detectar tipo de fuente ──────────────────────────────────
# Fuente API Socrata: tiene 'nombre_municipio' o 'c_digo_municipio'
# Fuente Geoportal Excel: tiene 'cod_mpio' y 'nom_mpio' y 'cod_dpto' y 'dpto'
# Fuente DANE censo: tiene columnas con 'municipio' en el nombre

def buscar_col(col_map, candidatos):
    '''Retorna el nombre ORIGINAL de la columna que contiene alguno de los candidatos.'''
    for c_lower, c_orig in col_map.items():
        for cand in candidatos:
            if cand in c_lower:
                return c_orig
    return None

# Mapeos posibles segun fuente
COL_COD_DEPTO    = buscar_col(col_map, ['cod_dpto', 'c_digo_dep', 'codigo_dep',
                                          'cod_dep', 'c_digo_dep', 'departamento_cod'])
COL_NOMBRE_DEPTO = buscar_col(col_map, ['dpto', 'nombre_dep', 'departamento',
                                          'nombre_departamento'])
COL_COD_MUN      = buscar_col(col_map, ['cod_mpio', 'c_digo_mun', 'codigo_mun',
                                          'cod_mun', 'c_digo_mun', 'municipio_cod'])
COL_NOMBRE_MUN   = buscar_col(col_map, ['nom_mpio', 'nombre_mun', 'nombre_municipio',
                                          'municipio', 'nom_municipio'])

print('Columnas detectadas automaticamente:')
print(f'  COD_DEPTO    : {COL_COD_DEPTO}')
print(f'  NOMBRE_DEPTO : {COL_NOMBRE_DEPTO}')
print(f'  COD_MUN      : {COL_COD_MUN}')
print(f'  NOMBRE_MUN   : {COL_NOMBRE_MUN}')

# ── Si alguna columna no se detecta, mostrar lista para asignacion manual ──
if not all([COL_COD_DEPTO, COL_NOMBRE_DEPTO, COL_COD_MUN, COL_NOMBRE_MUN]):
    print()
    print('AVISO: No se detectaron todas las columnas automaticamente.')
    print('Columnas disponibles (copia el nombre exacto):')
    for i, c in enumerate(df_divipola_raw.columns):
        print(f'  [{i:2d}]  {c!r}')
    print()
    print('Asigna manualmente las que falten y re-ejecuta esta celda:')
    print("  COL_COD_DEPTO    = 'nombre_columna'")
    print("  COL_NOMBRE_DEPTO = 'nombre_columna'")
    print("  COL_COD_MUN      = 'nombre_columna'")
    print("  COL_NOMBRE_MUN   = 'nombre_columna'")
else:
    # ── Construir DataFrame canonico ─────────────────────────
    def limpiar(serie):
        return serie.astype(str).str.strip().str.upper()

    df_divipola = pd.DataFrame({
        'cod_depto'     : df_divipola_raw[COL_COD_DEPTO].astype(str).str.strip().str.zfill(2),
        'nombre_depto'  : limpiar(df_divipola_raw[COL_NOMBRE_DEPTO]),
        'cod_divipola'  : df_divipola_raw[COL_COD_MUN].astype(str).str.strip().str.zfill(5),
        'nombre_oficial': limpiar(df_divipola_raw[COL_NOMBRE_MUN]),
    })

    # Tambien recuperar latitud/longitud si existen en la fuente
    # (el Geoportal DANE las incluye: columnas 'latitud' y 'longitud')
    col_lat = buscar_col(col_map, ['latitud', 'lat'])
    col_lon = buscar_col(col_map, ['longitud', 'lon'])
    if col_lat and col_lon:
        df_divipola['latitud_divipola']  = pd.to_numeric(
            df_divipola_raw[col_lat].astype(str).str.replace(',','.'), errors='coerce'
        )
        df_divipola['longitud_divipola'] = pd.to_numeric(
            df_divipola_raw[col_lon].astype(str).str.replace(',','.'), errors='coerce'
        )
        print(f'\nBonus: coordenadas lat/lon incluidas desde {FUENTE_USADA}')
        print(f'   (usadas como latitud_ref / longitud_ref para NASA POWER)')

    # Eliminar filas con codigo invalido (encabezados residuales, totales, etc.)
    df_divipola = df_divipola[
        df_divipola['cod_divipola'].str.match(r'^\d{5}$')
    ].drop_duplicates(subset=['cod_divipola']).reset_index(drop=True)

    # Guardar raw en Drive (util para re-ejecuciones sin conexion)
    try:
        if not os.path.exists(ARCHIVO_DIVIPOLA_RAW):
            df_divipola_raw.to_excel(ARCHIVO_DIVIPOLA_RAW, index=False, engine='openpyxl')
            print(f'Raw guardado en Drive: {ARCHIVO_DIVIPOLA_RAW}')
    except Exception as e_save:
        print(f'Aviso: no se pudo guardar raw ({e_save}) — continua sin problema')

    print(f'\nOK df_divipola construido exitosamente')
    print(f'   Fuente     : {FUENTE_USADA}')
    print(f'   Municipios : {len(df_divipola):,} (esperado: ~1.122)')
    print(f'   Columnas   : {df_divipola.columns.tolist()}')
    display(df_divipola.head(8))


---
## CELDA 4-MANUAL — Fallback si la CELDA 4 fallo

> Ejecutar SOLO si la descarga automatica fallo en CELDA 4.

In [ ]:
# ============================================================
# CELDA 4-MANUAL — Ejecutar SOLO si la CELDA 4 fallo
# Construye df_divipola directamente desde la API JSON de Socrata
# con los nombres de columna conocidos del recurso gdxc-w37w
# ============================================================

# Si la CELDA 4 no pudo descargar el archivo, este bloque construye
# df_divipola_raw directamente desde la respuesta JSON de la API
# sin necesidad de guardar ningun archivo en Drive.

print('Descargando DIVIPOLA desde API Socrata (gdxc-w37w)...')

URL_SOCRATA_2 = 'https://www.datos.gov.co/resource/gdxc-w37w.json'
registros_manual = []
offset_m = 0
limit_m  = 1000

while True:
    resp_m = requests.get(
        URL_SOCRATA_2,
        params={'$limit': limit_m, '$offset': offset_m},
        timeout=60
    )
    resp_m.raise_for_status()
    chunk_m = resp_m.json()
    if not chunk_m:
        break
    registros_manual.extend(chunk_m)
    print(f'  offset={offset_m:,}  registros={len(chunk_m):,}  total={len(registros_manual):,}')
    offset_m += limit_m
    if len(chunk_m) < limit_m:
        break
    time.sleep(0.3)

df_divipola_raw = pd.DataFrame(registros_manual)
FUENTE_USADA = 'API Socrata (manual)'

print(f'\nOK {len(df_divipola_raw):,} municipios descargados')
print(f'Columnas: {df_divipola_raw.columns.tolist()}')
display(df_divipola_raw.head(5))
print()
print('Ahora ejecuta la CELDA 5 para construir df_divipola canonico.')


---
## CELDA 6 — Extraccion de nombres unicos de cada fuente

> Se extraen muestras de cada fuente. No descarga datos completos.

In [ ]:
# ============================================================
# EXTRACCION DE NOMBRES UNICOS — Por fuente
# ============================================================

nombres_por_fuente = {}

# 1. Nombres desde EVA-UPRA
print('1) Extrayendo nombres desde EVA-UPRA...')
try:
    resp_eva = requests.get(URL_EVA, timeout=120)
    resp_eva.raise_for_status()
    df_eva_raw = pd.read_excel(
        BytesIO(resp_eva.content), sheet_name=0,
        skiprows=INI_FILA_EVA - 1, engine='openpyxl'
    )
    df_eva_raw = df_eva_raw.applymap(fix_encoding)
    cols_obj = df_eva_raw.select_dtypes(include='object').columns
    df_eva_raw[cols_obj] = df_eva_raw[cols_obj].apply(normalizar_texto)
    mascara = (
        df_eva_raw.get('Cultivo', pd.Series()).str.contains(CULTIVO_UPRA, na=False) &
        df_eva_raw.get('Desagregacion cultivo', pd.Series()).str.contains(DESAGR_UPRA, na=False) &
        df_eva_raw.get('Anio', pd.Series()).between(ANIO_INI_UPRA, ANIO_FIN_UPRA)
    )
    # Intentar tambien columnas con acentos
    if not mascara.any():
        for col in df_eva_raw.columns:
            if 'CULT' in str(col).upper():
                mascara = df_eva_raw[col].str.contains(CULTIVO_UPRA, na=False)
                if mascara.any():
                    break
    df_eva_f = df_eva_raw[mascara]
    col_mun_eva = next((c for c in df_eva_f.columns
                         if 'MUNICIPIO' in str(c).upper()), None)
    if col_mun_eva:
        nombres_por_fuente['upra'] = set(df_eva_f[col_mun_eva].dropna().unique())
        print(f'   OK UPRA: {len(nombres_por_fuente["upra"])} municipios unicos')
    else:
        print('   AVISO: No se encontro columna Municipio en UPRA')
except Exception as e:
    print(f'   AVISO Error UPRA: {e}')

# 2. Nombres desde catalogo estaciones IDEAM
print('\n2) Extrayendo nombres desde catalogo estaciones IDEAM...')
try:
    resp_est = requests.get(URL_ESTACIONES,
                             params={'$limit': LIMIT_ESTACIONES}, timeout=120)
    resp_est.raise_for_status()
    df_est = pd.DataFrame(resp_est.json())
    if 'municipio' in df_est.columns:
        df_est['municipio'] = normalizar_texto(df_est['municipio'])
        nombres_por_fuente['ideam_estaciones'] = set(df_est['municipio'].dropna().unique())
        print(f'   OK IDEAM Estaciones: {len(nombres_por_fuente["ideam_estaciones"])} municipios')
except Exception as e:
    print(f'   AVISO Error IDEAM Estaciones: {e}')

# 3. Nombres desde APIs climaticas IDEAM (muestras)
print('\n3) Extrayendo nombres desde APIs climaticas IDEAM (muestras)...')
mapeo_fuentes = {
    'temperatura'  : 'ideam_temperatura',
    'precipitacion': 'ideam_precipitacion',
    'humedad'      : 'ideam_humedad',
}
for key_api, key_fuente in mapeo_fuentes.items():
    url_api = APIS_IDEAM[key_api]
    try:
        resp_api = requests.get(url_api,
                                  params={'$limit': LIMIT_MUESTRA}, timeout=60)
        resp_api.raise_for_status()
        data_api = resp_api.json()
        if data_api:
            df_api = pd.DataFrame(data_api)
            if 'municipio' in df_api.columns:
                df_api['municipio'] = normalizar_texto(df_api['municipio'])
                nombres_por_fuente[key_fuente] = set(df_api['municipio'].dropna().unique())
                print(f'   OK IDEAM {key_api}: {len(nombres_por_fuente[key_fuente])} nombres')
            else:
                print(f'   AVISO {key_api}: sin campo municipio')
        else:
            print(f'   AVISO {key_api}: respuesta vacia')
        time.sleep(0.5)
    except Exception as e:
        print(f'   AVISO Error {key_api}: {e}')

print('\nResumen de nombres por fuente:')
for fuente, nombres in nombres_por_fuente.items():
    print(f'   {fuente:<30}: {len(nombres):>4} nombres unicos')


---
## CELDA 7 — Fuzzy matching: homologacion automatica contra DIVIPOLA

In [ ]:
# ============================================================
# FUZZY MATCHING — Homologacion automatica contra DIVIPOLA
# ============================================================

print('Iniciando fuzzy matching por fuente...')
print(f'   Umbral automatico  : >= {UMBRAL_FUZZY_AUTO}  -> Automatica')
print(f'   Umbral manual      : {UMBRAL_FUZZY_MAN}-{UMBRAL_FUZZY_AUTO-1} -> Revision manual')
print(f'   Sin match          : < {UMBRAL_FUZZY_MAN}')
print()

resultados_por_fuente = {}

FUENTES_COLS = {
    'upra'               : 'nombre_upra',
    'ideam_estaciones'   : 'nombre_ideam_estaciones',
    'ideam_temperatura'  : 'nombre_ideam_temp',
    'ideam_precipitacion': 'nombre_ideam_prec',
    'ideam_humedad'      : 'nombre_ideam_hr',
}

for fuente_key, col_destino in FUENTES_COLS.items():
    nombres_f = sorted(nombres_por_fuente.get(fuente_key, set()))
    if not nombres_f:
        print(f'AVISO {fuente_key}: sin nombres — se omite')
        continue
    print(f'\nProcesando: {fuente_key} ({len(nombres_f)} nombres)...')
    df_match = homologar_nombres(
        nombres_f, df_divipola,
        umbral_auto=UMBRAL_FUZZY_AUTO,
        umbral_manual=UMBRAL_FUZZY_MAN
    )
    df_match.rename(columns={'nombre_fuente': col_destino}, inplace=True)
    resultados_por_fuente[fuente_key] = (df_match, col_destino)
    resumen = df_match['metodo'].value_counts()
    for metodo, n in resumen.items():
        emoji = 'OK' if metodo == 'Exacto' else ('AVISO' if 'Revision' in str(metodo) else '-')
        print(f'   {emoji} {metodo:<25}: {n:3d}')

print('\nFuzzy matching completado')


---
## CELDA 8 — Construccion de la tabla maestra de homologacion

In [ ]:
# ============================================================
# CONSTRUCCION TABLA MAESTRA DE HOMOLOGACION
# ============================================================

df_maestra = df_divipola.copy()
df_maestra['es_municipio_arrocero'] = False

for col in ['nombre_upra','nombre_ideam_estaciones','nombre_ideam_temp',
            'nombre_ideam_prec','nombre_ideam_hr']:
    df_maestra[col]             = None
    df_maestra[f'score_{col}']  = None
    df_maestra[f'metodo_{col}'] = None

# Poblar desde resultados del fuzzy matching
for fuente_key, (df_r, col_destino) in resultados_por_fuente.items():
    for _, row in df_r.iterrows():
        if pd.notna(row.get('cod_divipola')) and row.get('metodo','') != 'Sin_match':
            mask = df_maestra['cod_divipola'] == row['cod_divipola']
            if mask.any():
                df_maestra.loc[mask, col_destino]             = row[col_destino]
                df_maestra.loc[mask, f'score_{col_destino}']  = row['score']
                df_maestra.loc[mask, f'metodo_{col_destino}'] = row['metodo']
                if fuente_key == 'upra':
                    df_maestra.loc[mask, 'es_municipio_arrocero'] = True

# Coordenadas desde catalogo de estaciones
df_maestra['latitud_ref']  = None
df_maestra['longitud_ref'] = None
try:
    df_est_full = pd.DataFrame(
        requests.get(URL_ESTACIONES,
                     params={'$limit': LIMIT_ESTACIONES}, timeout=120).json()
    )
    for col in ['latitud','longitud']:
        if col in df_est_full.columns:
            df_est_full[col] = pd.to_numeric(df_est_full[col], errors='coerce')
    if 'municipio' in df_est_full.columns:
        df_est_full['municipio'] = normalizar_texto(df_est_full['municipio'])
        df_coords = (
            df_est_full.dropna(subset=['latitud','longitud'])
            .groupby('municipio', as_index=False)
            .agg(lat=('latitud','median'), lon=('longitud','median'))
        )
        for _, row in df_coords.iterrows():
            mask_c = df_maestra['nombre_ideam_estaciones'] == row['municipio']
            if mask_c.any():
                df_maestra.loc[mask_c, 'latitud_ref']  = row['lat']
                df_maestra.loc[mask_c, 'longitud_ref'] = row['lon']
        print(f'OK Coordenadas: {df_maestra["latitud_ref"].notna().sum()} municipios con lat/lon')
except Exception as e:
    print(f'AVISO Error coordenadas: {e}')

# Estado de cobertura
FUENTES_BASE = ['nombre_upra','nombre_ideam_estaciones',
                'nombre_ideam_temp','nombre_ideam_prec','nombre_ideam_hr']

def calcular_estado(row):
    n = sum(1 for c in FUENTES_BASE if pd.notna(row.get(c)))
    return 'Completo' if n == 5 else ('Parcial' if n >= 3 else ('Parcial_bajo' if n >= 1 else 'Sin_datos'))

def calcular_fuentes(row):
    et = {'nombre_upra':'upra','nombre_ideam_estaciones':'est',
          'nombre_ideam_temp':'temp','nombre_ideam_prec':'prec','nombre_ideam_hr':'hr'}
    return ','.join(v for k,v in et.items() if pd.notna(row.get(k)))

df_maestra['estado_cobertura']    = df_maestra.apply(calcular_estado, axis=1)
df_maestra['fuentes_disponibles'] = df_maestra.apply(calcular_fuentes, axis=1)
df_maestra['fecha_revision']      = pd.Timestamp.now().strftime('%Y-%m-%d')
df_maestra['observaciones']       = ''

cols_ord = [
    'cod_divipola','nombre_oficial','cod_depto','nombre_depto',
    'nombre_upra','nombre_ideam_estaciones',
    'nombre_ideam_temp','nombre_ideam_prec','nombre_ideam_hr',
    'latitud_ref','longitud_ref',
    'estado_cobertura','fuentes_disponibles',
    'es_municipio_arrocero','fecha_revision','observaciones'
]
df_maestra = df_maestra[[c for c in cols_ord if c in df_maestra.columns]].reset_index(drop=True)

reporte_df(df_maestra, 'df_maestra — Tabla Maestra de Homologacion')
display(df_maestra[df_maestra['es_municipio_arrocero']].head(10))


---
## CELDA 9 — Diagnostico de cobertura y casos de revision manual

In [ ]:
# ============================================================
# DIAGNOSTICO DE COBERTURA Y CASOS DE REVISION MANUAL
# ============================================================

print('='*62)
print('  DIAGNOSTICO DE COBERTURA POR FUENTE')
print('='*62)

for col_base in FUENTES_BASE:
    n_con = df_maestra[col_base].notna().sum()
    n_sin = df_maestra[col_base].isna().sum()
    pct   = n_con / len(df_maestra) * 100
    col_m = f'metodo_{col_base}'
    print(f'\n  {col_base}')
    print(f'     Con datos : {n_con:,} ({pct:.1f}%)')
    print(f'     Sin datos : {n_sin:,} ({100-pct:.1f}%)')
    if col_m in df_maestra.columns:
        for metodo, n in df_maestra[col_m].value_counts().items():
            print(f'     - {metodo:<30}: {n:3d}')

print('\nEstado de cobertura global:')
display(df_maestra['estado_cobertura'].value_counts().reset_index().rename(
    columns={'estado_cobertura': 'Estado', 'count': 'N_Municipios'}
))

frames_rev = []
for col_base in FUENTES_BASE:
    col_m = f'metodo_{col_base}'
    if col_m in df_maestra.columns:
        mask_rev = df_maestra[col_m].isin(['Revision_manual','Sin_match'])
        df_sub = df_maestra[mask_rev][
            ['cod_divipola','nombre_oficial','nombre_depto',
             col_base, col_m, f'score_{col_base}']
        ].copy()
        df_sub.columns = ['cod_divipola','nombre_oficial','nombre_depto',
                          'nombre_fuente','metodo','score']
        df_sub['fuente'] = col_base
        frames_rev.append(df_sub)

df_revision = pd.concat(frames_rev, ignore_index=True) if frames_rev else pd.DataFrame()
df_revision['homologacion_manual'] = ''
df_revision['observaciones']       = ''

print(f'\nCasos para revision manual: {len(df_revision)}')
if len(df_revision) > 0:
    display(df_revision.sort_values(['fuente','score'], ascending=[True,False]).head(30))
    print('-> Completar columna homologacion_manual en hoja Casos_revision del Excel')


---
## CELDA 10 — Visualizacion: Dashboard de cobertura

In [ ]:
# ============================================================
# VISUALIZACION — Dashboard de cobertura municipal
# ============================================================

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

col_map = {'Completo': '#4CAF50', 'Parcial': '#FFC107',
           'Parcial_bajo': '#FF9800', 'Sin_datos': '#F44336'}

# Grafica 1: Estado de cobertura (pastel)
estados = df_maestra['estado_cobertura'].value_counts()
axes[0].pie(
    estados.values, labels=estados.index,
    colors=[col_map.get(e,'#9E9E9E') for e in estados.index],
    autopct='%1.1f%%', startangle=90, pctdistance=0.8
)
axes[0].set_title('Estado de cobertura\n(todos los municipios Colombia)',
                  fontsize=11, fontweight='bold')

# Grafica 2: Municipios arroceros por estado
df_arr = df_maestra[df_maestra['es_municipio_arrocero']]
estado_arr = df_arr['estado_cobertura'].value_counts()
axes[1].bar(estado_arr.index, estado_arr.values,
            color=[col_map.get(e,'#9E9E9E') for e in estado_arr.index],
            edgecolor='white')
axes[1].set_title('Municipios arroceros\npor estado de cobertura',
                  fontsize=11, fontweight='bold')
axes[1].set_ylabel('N municipios')
axes[1].tick_params(axis='x', rotation=20)
axes[1].grid(axis='y', alpha=0.3)

# Grafica 3: Cobertura por departamento (top 15)
df_dep = (
    df_maestra[df_maestra['estado_cobertura'].isin(['Completo','Parcial'])]
    .groupby('nombre_depto').size().nlargest(15).reset_index(name='n')
)
axes[2].barh(df_dep['nombre_depto'], df_dep['n'], color='#1565C0', alpha=0.8)
axes[2].set_title('Top 15 departamentos\npor municipios con cobertura',
                  fontsize=11, fontweight='bold')
axes[2].set_xlabel('N municipios con datos')
axes[2].grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()

print(f'\nResumen ejecutivo:')
print(f'   Total municipios DIVIPOLA       : {len(df_maestra):,}')
print(f'   Municipios arroceros (UPRA)     : {df_maestra["es_municipio_arrocero"].sum():,}')
print(f'   Con cobertura completa (5 ftes) : {(df_maestra["estado_cobertura"]=="Completo").sum():,}')
print(f'   Con cobertura parcial           : {df_maestra["estado_cobertura"].isin(["Parcial","Parcial_bajo"]).sum():,}')
print(f'   Sin datos en ninguna fuente     : {(df_maestra["estado_cobertura"]=="Sin_datos").sum():,}')
print(f'   Casos pendientes revision manual: {len(df_revision):,}')


---
## CELDA 11 — Guardar tabla maestra y reporte de cobertura

In [ ]:
# ============================================================
# GUARDAR — Tabla maestra y archivos de salida
# ============================================================

guardar_excel(
    {
        'Todos_municipios': df_maestra,
        'Solo_arroceros'  : df_maestra[df_maestra['es_municipio_arrocero']].copy(),
        'Casos_revision'  : df_revision if len(df_revision) > 0 else pd.DataFrame(
            columns=['cod_divipola','nombre_oficial','nombre_depto',
                     'nombre_fuente','metodo','score','fuente',
                     'homologacion_manual','observaciones']
        ),
    },
    ARCHIVO_HOMOLOG_XLSX,
    nombre_log='00_DF_Homologacion_Municipios.xlsx'
)

df_maestra.to_parquet(ARCHIVO_HOMOLOG_PARQ, index=False, engine='pyarrow')
size_mb = os.path.getsize(ARCHIVO_HOMOLOG_PARQ) / 1_048_576
print(f'OK 00_DF_Homologacion_Municipios.parquet -> {ARCHIVO_HOMOLOG_PARQ}')
print(f'   Shape : {df_maestra.shape[0]:,} x {df_maestra.shape[1]}')
print(f'   Tamano: {size_mb:.2f} MB')

df_reporte = (
    df_maestra.groupby('nombre_depto')
    .agg(
        total_municipios     = ('cod_divipola', 'count'),
        con_upra             = ('nombre_upra', lambda x: x.notna().sum()),
        con_temp             = ('nombre_ideam_temp', lambda x: x.notna().sum()),
        con_prec             = ('nombre_ideam_prec', lambda x: x.notna().sum()),
        con_hr               = ('nombre_ideam_hr', lambda x: x.notna().sum()),
        cobertura_completa   = ('estado_cobertura', lambda x: (x=='Completo').sum()),
        municipios_arroceros = ('es_municipio_arrocero', 'sum'),
    )
    .reset_index()
    .sort_values('total_municipios', ascending=False)
)
guardar_excel(df_reporte, ARCHIVO_REPORTE, nombre_log='00_Reporte_Cobertura.xlsx')


---
## CELDA 12 — Resumen de ejecucion del modulo

In [ ]:
# ============================================================
# RESUMEN FINAL — MOD-00 completado
# ============================================================

print('='*62)
print('  MOD-00 — Homologacion de Municipios COMPLETADO')
print('='*62)

print('\nArchivos generados:')
archivos = [
    ('data/raw/',       '00_raw_DIVIPOLA.xlsx',                 'Listado DIVIPOLA del DANE (sin modificar)'),
    ('data/processed/', '00_DF_Homologacion_Municipios.xlsx',   'Tabla maestra (3 hojas: Todos, Arroceros, Revision)'),
    ('data/processed/', '00_DF_Homologacion_Municipios.parquet','Tabla maestra en Parquet para joins en memoria'),
    ('data/processed/', '00_Reporte_Cobertura.xlsx',            'Cobertura por departamento'),
]
for carpeta, archivo, desc in archivos:
    print(f'   {carpeta}{archivo}')
    print(f'      {desc}')

print('\nCampo clave generado: cod_divipola')
print('   Llave canonica universal DIVIPOLA del DANE')
print('   Usado en MOD-01, MOD-02, MOD-03, MOD-04, MOD-05, MOD-06')

print('\nEstadisticas:')
print(f'   Total municipios DIVIPOLA         : {len(df_maestra):,}')
print(f'   Municipios arroceros (arroz riego): {df_maestra["es_municipio_arrocero"].sum():,}')
print(f'   Cobertura completa (5 fuentes)    : {(df_maestra["estado_cobertura"]=="Completo").sum():,}')
print(f'   Casos pendientes revision manual  : {len(df_revision):,}')

print('\nProximos pasos:')
print('   1. Revisar hoja Casos_revision en Excel y completar homologacion_manual')
print('   2. Ejecutar MOD-01 (usa cod_divipola como nueva llave de join)')
print('   3. Ejecutar MOD-02 -> MOD-03 -> MOD-04 -> MOD-05 -> MOD-06')
print('='*62)
